In [1]:
import selenium 
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from splinter import Browser
from bs4 import BeautifulSoup as soup
import re
import pandas as pd
import matplotlib.pyplot as plt
import time
import requests
import PyPDF2
from urllib.parse import urljoin
import os
import json

In [2]:
opera_path = "/Applications/Opera.app/Contents/MacOS/Opera"  # change this path
driver_path = "/path/to/chromedriver"
pdf_storage = "/all pds"
dest_folder = "Data"
i = 0

In [3]:
#dataPoints = []
#driver = webdriver.Safari()

In [4]:
def htmlData(url):

    response = requests.get(url)
    html_content = response.text
    html = soup(html_content, "html.parser")
    #print(html)
     # Get title
    
    title_tag = html.find("title")
    title = title_tag.text.strip() if title_tag else None

    # Get all paragraphs
    paragraphs = html.find_all('p')
    paragraph_texts = [p.get_text(strip=True) for p in paragraphs]

    # Get all spans
    spans = html.find_all('span')
    spans_texts = [s.get_text(strip=True) for s in spans]
    
    # Get all links
    links = [a.get('href') for a in html.find_all('a') if a.get('href')]

    #Get list if possible 
    lists = html.find_all('ul')
    extraInfo = []
    for ul in lists:
        extraInfo = (ul.get_text(strip=True, separator=', '))

    correctedSpan = spans_texts[:10]
    correctedLinks = links[:20]

    
    # Find all links
    pdf_links = []
    for tag in html.find_all(True):  # True matches all tags
        for attr in ["href", "src"]:
            if tag.has_attr(attr):
                link = tag[attr]
                if "pdf" in link.lower():  # check if link contains pdf
                    full_url = urljoin(url, link)
                    pdf_links.append(full_url)

    
    pdf_text = ""
    pdf_metadata = ""
    if (is_pdf_url(url)):
        file_path = pdfDownload([url])
        pdf_text = getDataPdf(file_path)
        pdf_metadata = extract_pdf_metadata(file_path)

    content = {
        "title" : title,
        "paragraphs" : paragraph_texts,
        "span" : correctedSpan,
        "extra info" : extraInfo,
        "links" : correctedLinks,
        "pdf" :  pdf_text,
        "metadata" : pdf_metadata}

    if not is_pdf_url(url):
        store_contents(content)
    
    return content

In [5]:
def is_pdf_url(url):
    try:
        response = requests.head(url, allow_redirects=True)
        content_type = response.headers.get("Content-Type", "").lower()
        if "application/pdf" in content_type:
            return True
        else:
            return False
    except requests.RequestException:
        return False

In [6]:
def pdfDownload(pdf_links):
    global i
    saved_pdfs = []
    
    for pdf_url in pdf_links:
            try: 
                print(f"Downloading {pdf_url}...")
                response = requests.get(pdf_url)
                
                if response.status_code == 200:
                    filename = f"document_{i}.pdf"
                    i += 1
                    filename = os.path.join(dest_folder, filename)
                    
                    with open(filename, "wb") as pdf_file:
                        pdf_file.write(response.content)
                    print(f"Saved as {filename}")
                    
                    saved_pdfs.append(filename)
                    
            except requests.RequestException as e:
                print(f"Failed to download {pdf_url}: {e}")
                
    return saved_pdfs

In [7]:
def getDataPdf(pdf_list):
    text = []
    for pdf in pdf_list:
        try:
            with open(pdf, "rb") as f:
                reader = PyPDF2.PdfReader(f)
                text = ""
                for page in reader.pages:
                    text += page.extract_text()
        except PyPDF2.errors.PdfReadError:
            print(f"Skipping {pdf}, not a valid PDF.")
    return text

In [8]:
def extract_pdf_metadata(pdf_list):
    metadata = {}
    for pdf in pdf_list:
        with open(pdf, "rb") as f:
            reader = PyPDF2.PdfReader(f)
            info = reader.metadata  # PyPDF2 >= 3.0
            if info:
                metadata['title'] = info.title
                metadata['author'] = info.author
                metadata['creator'] = info.creator
                metadata['producer'] = info.producer
                metadata['subject'] = info.subject
                metadata['creation_date'] = info.creation_date
    return metadata

In [9]:
def store_contents(content):
    
    dest_folder = "Data"
    os.makedirs(dest_folder, exist_ok=True)
    
    contentstoS = json.dumps(content)
    filename = content.get("title", "untitled") + ".txt"
    file_path = os.path.join(dest_folder, filename)

    with open(file_path, "w", encoding="utf-8") as f:
        f.write(contentstoS)
    print(f"Saved as {filename}")
        

In [10]:
test = htmlData('https://science.nasa.gov/mars/facts/#h-size-and-distance')
test2 = htmlData('https://www.nasa.gov/wp-content/uploads/2015/01/601288main_NASAsContributionsToAeronauticsVolume1-ebook.pdf?emrc=f9cfaa')
test3 = htmlData('https://dataverse.jpl.nasa.gov/api/access/datafile/4047?gbrecs=true')

Saved as Mars: Facts - NASA Science.txt
Saved as Data/document_0.pdf
Saved as Data/document_1.pdf


In [11]:
test
test2
test3

{'title': None,
 'paragraphs': [],
 'span': [],
 'extra info': [],
 'links': [],
 'pdf': " \n \nAm e r i c a n  I n s t i t u t e  o f  A e r o n a u t i c s  a n d  A s t r o n a u t i c s  \n \n 1D u a l  U s e  o f  P a c k a g i n g  o n  t h e  M o o n :  L o g i s t i c s - 2 - L i v i n g  \nA .  S c o t t  H o w e ,  P h D 1 \nR o b e r t  H o w a r d ,  P h D 2 \nN A S A  L u n a r  S u r f a c e  S y s t em s  P r o j e c t  ( L S S P )  H a b i t a t i o n  T e am  m em b e r s 3 \nT h i s  p a p e r  d e s c r i b e s  a  m o d u l a r  p a c k a g i n g  s y s t em  f o r  l o g i s t i c s  t h a t  c a n  b e  r e c o n f i g u r e d  \ni n t o  i n t e r n a l  o u t f i t t i n g  f o r  a  l u n a r  o u t p o s t ,  i n c l u d i n g  d e s k s ,  c h a i r s ,  p a r t i t i o n s ,  c a b i n e t s ,  a n d  \nr a d i a t i o n  s h i e l d i n g .  L o g i s t i c s  i n c l u d e  c l o t h e s ,  e q u i pm e n t ,  f o o d ,  a n d  o t h e r  c o n s um a b l 